# 01 - Master List Analysis

## Purpose

The purpose of this notebook is to analyze the supplier master list used in the manual import process. The analysis focuses on:

- File structure
- Number of rows
- Column names
- Missing values
- Duplicate entries
- Potential issues affecting the ETL pipeline

## Imports

In [ ]:
import duckdb

## File Configuration

In [ ]:
SUPPLIER = "snickers"

MASTER_LIST_PATH = (
    f"../data/{SUPPLIER}/master_list/Snickers_Masterlista_2026.csv"
)

## Raw File Inspection

Inspect the raw file before loading it into pandas or other tools. This helps identify:

- Encoding
- Delimiter
- Header rows
- File structure
- Multiline fields
- Unexpected formatting issues

In [ ]:
with open(MASTER_LIST_PATH, encoding="utf-8") as f:
    for _ in range(5):
        print(repr(f.readline()))

## Load Dataset

Load the supplier master list using DuckDB.

In [ ]:
duckdb.sql(f"""
CREATE OR REPLACE TABLE master_list AS 
SELECT *
FROM read_csv_auto('{MASTER_LIST_PATH}')
""")

## Dataset Overview

Preview the first rows to understand the available columns and values.

In [ ]:
duckdb.sql("SELECT * FROM master_list LIMIT 10").df()

## Dataset Statistics

In [ ]:
duckdb.sql("""
SELECT COUNT(*) AS total_rows
FROM master_list
""").df()

## Schema Analysis

In [ ]:
duckdb.sql("DESCRIBE master_list").df()

## Data Quality Checks

In [ ]:
duckdb.sql("""
SELECT
    COUNT(*) FILTER (WHERE Modell IS NULL) AS missing_modell,
    COUNT(*) FILTER (WHERE Produktnamn IS NULL) AS missing_product_name,
    COUNT(*) FILTER (WHERE Status IS NULL) AS missing_status,
    COUNT(*) FILTER (WHERE Kommentar IS NULL) AS missing_comments
FROM master_list
""").df()

In [ ]:
duckdb.sql("""
SELECT
    Modell,
    COUNT(*) AS occurrences
FROM master_list
GROUP BY Modell
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
""").df()

In [ ]:
duckdb.sql("""SELECT Status, COUNT(*) AS count
        FROM master_list
        GROUP BY Status
        ORDER BY count DESC
        """).df()

In [ ]:
duckdb.sql("""
        SELECT COUNT(*) AS different_prices
        FROM master_list
        WHERE "Lägsta RRP 2026" != "Högsta RRP 2026"
        """).df()

## Model Key Validation

Check whether `Modell` can be used as a unique identifier.

In [ ]:
duckdb.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT Modell) as unique_models
FROM master_list
""").df()

### Result

- Modell is unique (600/600)

## Findings

- The supplier master list contains 600 rows.
- Status drives the import workflow.
- One product is discontinued.
- Kommentar is mostly empty.
- Lowest and Highest RRP are identical.
- Modell can be used as a candidate key.

## Open Questions

- Are all models present in the latest price list?
- Have any prices changed?
- Are there additional discontinued products?
- Are all products represented in the XML feed?